# 🏆 FDS Challenge: Pokémon Battle Winner Prediction (Gen 2 OU)

Welcome to the complete, competitive notebook for the **FDS Pokémon Battles Prediction 2026** competition!

The objective is to predict whether **Player 1 wins the match (`player_won` = 1)** or loses (`player_won` = 0) in competitive **Generation 2 OverUsed (OU)** Pokémon battles, based on the initial team compositions and the **first 30 active battle turns** (`train.jsonl` and `test.jsonl`).

### 📌 Pipeline Architecture
1. **Data Loading & Dynamic Path Resolution**: Automatic portability across Kaggle environments and local directories.
2. **Modular 3-Pillar Feature Engineering**:
   * *Pillar 1*: True team HP (accounting for unrevealed reserve Pokémon at 100% HP).
   * *Pillar 2*: Turn 30 field state & Gen 2 elemental type advantage.
   * *Pillar 3*: Late-game momentum (HP trajectory between turns 20-30) and tactical metagame factors (Snorlax alive, Spikes hazard control).
3. **Multi-Model Assembly (GBDT Trinity)**:
   * `HistGradientBoostingClassifier` (Scikit-Learn histogram binning)
   * `LGBMClassifier` (LightGBM leaf-wise growth)
   * `XGBClassifier` (XGBoost depth-wise with exact Hessian gradients)
4. **Out-Of-Fold (OOF) Validation & Decision Threshold Calibration**:
   * Stratified 5-Fold Cross-Validation preventing data leakage.
   * Optimal probability blending weights solved numerically via SciPy.
   * Decision threshold calibration directly maximizing classification Accuracy.
5. **Submission Generation & Validation (`submission.csv`)**.

## 1. Data Loading & Dynamic Path Resolution

To ensure this notebook runs **seamlessly without errors** both in Kaggle environments (with paths under `/kaggle/input/...`) and locally (in `.`), we implement a dynamic `resolve_path()` helper.

We load:
* `train.jsonl`: 10,000 battles with ground-truth `player_won` labels and 30-turn timelines.
* `test.jsonl`: 5,000 battles for leaderboard prediction.
* `type_chart_gen2.csv`: Official Generation 2 type effectiveness matrix (111 multipliers).

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Automatic path resolution (works both on Kaggle and locally)
def resolve_path(filename):
    kaggle_paths = [
        Path("/kaggle/input/competitions/fds-pokemon-battles-prediction-2026") / filename,
        Path("/kaggle/input/fds-pokemon-battles-prediction-2026") / filename,
        Path(".") / filename
    ]
    for p in kaggle_paths:
        if p.exists():
            return p
    return Path(filename)

train_path = resolve_path('train.jsonl')
test_path = resolve_path('test.jsonl')
type_chart_path = resolve_path('type_chart_gen2.csv')

print(f"Loading training data from {train_path}...")
train_data = []
with open(train_path, 'r', encoding='utf-8') as f:
    for line in f:
        train_data.append(json.loads(line))
print(f"> Loaded {len(train_data)} training battles.")

print(f"Loading test data from {test_path}...")
test_data = []
with open(test_path, 'r', encoding='utf-8') as f:
    for line in f:
        test_data.append(json.loads(line))
print(f"> Loaded {len(test_data)} test battles.")

# Load Gen 2 type effectiveness chart
type_chart_df = pd.read_csv(type_chart_path)
type_chart = {}
for _, row in type_chart_df.iterrows():
    type_chart[(str(row['attacking_type']).lower(), str(row['defending_type']).lower())] = float(row['multiplier'])
print(f"> Loaded type chart with {len(type_chart)} effectiveness multipliers.")

# Inspect the first battle
print("\nFirst battle preview:")
print(f"Battle ID: {train_data[0]['battle_id']}, Player 1 Won: {train_data[0]['player_won']}")
print(f"P1 Team: {[p['name'] for p in train_data[0].get('p1_team_details', [])]}")
print(f"P2 Lead: {train_data[0].get('p2_lead_details', {}).get('name')}")
print(f"Number of timeline turns: {len(train_data[0].get('battle_timeline', []))}")

Loading training data from train.jsonl...
> Loaded 10000 training battles.
Loading test data from test.jsonl...
> Loaded 5000 test battles.
> Loaded type chart with 110 effectiveness multipliers.

First battle preview:
Battle ID: 0, Player 1 Won: 0
P1 Team: ['tyranitar', 'typhlosion', 'golem', 'umbreon', 'alakazam', 'raikou']
P2 Lead: forretress
Number of timeline turns: 30


## 2. Feature Engineering: The 3 Pillars of Pokémon Gen 2 OU

The initial baseline notebook only considered mean base stats (6 raw features), yielding barely 51.6% accuracy (equivalent to a random coin flip).

Rather than adding dozens of noisy and redundant columns, we isolate **essential, high-signal features** organized around the **3 decisive metagame pillars**:
1. **Pillar 1**: True Team State (remaining HP across all 6 team members + 100% for unrevealed bench reserves).
2. **Pillar 2**: Turn 30 Field State (active Pokémon HP and Gen 2 elemental type advantage).
3. **Pillar 3**: Late-Game Momentum (HP trajectory in turns 20-30) and key tactical dynamics (`Spikes` hazard control, Snorlax alive status).

### 2.1 Mapping Species → Elemental Types & Effectiveness
In the battle timeline (`battle_timeline`), each turn records the active Pokémon name (e.g. `"typhlosion"`), but **not** its elemental types.
We build:
* `build_species_types_map()`: scans the dataset to map every Pokémon species to its primary and secondary types.
* `get_type_mult()`: computes the compound damage multiplier between attacker and defender types according to Gen 2 mechanics.

In [ ]:
def build_species_types_map(data):
    """
    Maps each Pokémon species to its primary and secondary types
    by scanning P1 team rosters and P2 leads.
    """
    species_types = {}
    for b in data:
        for p in b.get('p1_team_details', []):
            if 'types' in p and p['name'] not in species_types:
                species_types[p['name']] = p['types']
        lead = b.get('p2_lead_details')
        if lead and 'types' in lead and lead['name'] not in species_types:
            species_types[lead['name']] = lead['types']
    return species_types

species_map = build_species_types_map(train_data)
print(f"> Mapped {len(species_map)} Pokémon species with their elemental types.")

def get_type_mult(atk_types, def_types):
    """
    Calculates total effectiveness multiplier between attacking and defending types
    (e.g., Fire vs Bug/Steel = 2.0 * 2.0 = 4.0x damage).
    """
    mult = 1.0
    for a in atk_types:
        if not a or a == 'notype': continue
        for d in def_types:
            if not d or d == 'notype': continue
            mult *= type_chart.get((str(a).lower(), str(d).lower()), 1.0)
    return mult

> Mapped 239 Pokémon species with their elemental types.


### 2.2 Extraction of Essential Features (The 3 Pillars)

Following Ponytail principles, we eliminated duplicate loops and low-signal noise. Feature extraction focuses on:
1. **Pillar 1 (True Team HP)**: Remaining HP of active/revealed Pokémon + 100% (1.0) for unrevealed bench reserves. The feature `total_hp_diff` is the **#1 predictive feature in the dataset**.
2. **Pillar 2 (Turn 30 State)**: Active Pokémon HP and Gen 2 type advantage computed via `species_map`.
3. **Pillar 3 (Momentum & Tactics)**: HP trend between turns 20-30, entry hazard control (`Spikes`), recovery moves (`Rest`), and Snorlax status (present in 82% of Gen 2 OU matches).

In [3]:
from tqdm.auto import tqdm
import numpy as np
import pandas as pd

HEAL_MOVES = {'rest', 'recover', 'softboiled', 'synthesis', 'moonlight', 'morningsun'}
SETUP_MOVES = {'curse', 'swordsdance', 'bellydrum', 'agility', 'growth'}

def create_simple_features(data):
    """
    High-signal essential feature extraction:
    - Base stats and lead matchup
    - Pillar 1: True team HP (including 100% bench reserves)
    - Pillar 2: Turn 30 field state (active HP & elemental matchup)
    - Pillar 3: Late-game momentum (t20-t30), Spikes control, and Snorlax factor
    """
    rows = []
    for b in tqdm(data, desc="Extracting essential features"):
        f = {'battle_id': b['battle_id']}
        if 'player_won' in b:
            f['player_won'] = int(b['player_won'])

        # 1. P1 Base Stats & Lead Matchup
        p1_team = b.get('p1_team_details', [])
        p1_names = {p['name'] for p in p1_team}
        for s in ['hp', 'atk', 'def', 'spa', 'spd', 'spe']:
            v = [p.get(f'base_{s}', 0) for p in p1_team]
            f[f'p1_mean_{s}'], f[f'p1_std_{s}'] = float(np.mean(v)), float(np.std(v))

        p2_lead = b.get('p2_lead_details', {})
        if p1_team and p2_lead:
            p1_l = p1_team[0]
            f['lead_spe_diff'] = float(p1_l.get('base_spe', 0) - p2_lead.get('base_spe', 0))
            f['lead_type_adv'] = get_type_mult(p1_l.get('types', []), p2_lead.get('types', [])) - get_type_mult(p2_lead.get('types', []), p1_l.get('types', []))

        # 2. Timeline Tracking (First 30 Turns)
        p1_hp = {n: 1.0 for n in p1_names}
        p2_hp = {}
        acts = {'p1_sw': 0, 'p2_sw': 0, 'p1_spikes': 0, 'p2_spikes': 0, 'p1_heals': 0, 'p2_heals': 0, 'p1_setups': 0, 'p2_setups': 0}
        p1_t20, p2_t20 = 1.0, 1.0

        tl = b.get('battle_timeline', [])
        for t in tl:
            s1, s2 = t.get('p1_pokemon_state', {}), t.get('p2_pokemon_state', {})
            m1, m2 = t.get('p1_move_details'), t.get('p2_move_details')
            if s1.get('name'):
                p1_hp[s1['name']] = s1.get('hp_pct', 1.0)
                if t.get('turn') == 20: p1_t20 = p1_hp[s1['name']]
            if s2.get('name'):
                p2_hp[s2['name']] = s2.get('hp_pct', 1.0)
                if t.get('turn') == 20: p2_t20 = p2_hp[s2['name']]

            # Unified action counters
            for m, prefix in [(m1, 'p1'), (m2, 'p2')]:
                if not m:
                    acts[f'{prefix}_sw'] += 1
                else:
                    mn = m.get('name', '')
                    if mn == 'spikes': acts[f'{prefix}_spikes'] += 1
                    elif mn in HEAL_MOVES: acts[f'{prefix}_heals'] += 1
                    elif mn in SETUP_MOVES: acts[f'{prefix}_setups'] += 1

        # Pillar 1: True Team HP (Active + 100% Unrevealed Reserves)
        p1_tot = sum(p1_hp.values())
        p2_tot = sum(p2_hp.values()) + max(0, 6 - len(p2_hp)) * 1.0
        f['total_hp_diff'] = float(p1_tot - p2_tot)
        f['p1_total_hp'] = float(p1_tot)
        f['p2_total_hp'] = float(p2_tot)
        f['alive_diff'] = float(sum(1 for h in p1_hp.values() if h > 0) - (sum(1 for h in p2_hp.values() if h > 0) + max(0, 6 - len(p2_hp))))

        # Pillar 2: Turn 30 Field State
        if tl:
            last = tl[-1]
            a1, a2 = last.get('p1_pokemon_state', {}), last.get('p2_pokemon_state', {})
            f['act_hp_diff'] = float(a1.get('hp_pct', 0.0) - a2.get('hp_pct', 0.0))
            f['act_hp_p1'] = float(a1.get('hp_pct', 0.0))
            f['act_hp_p2'] = float(a2.get('hp_pct', 0.0))
            f['act_type_adv'] = float(get_type_mult(species_map.get(a1.get('name', ''), []), species_map.get(a2.get('name', ''), [])) - get_type_mult(species_map.get(a2.get('name', ''), []), species_map.get(a1.get('name', ''), [])))
            f['act_status_diff'] = float(a2.get('status') not in (None, 'nostatus')) - float(a1.get('status') not in (None, 'nostatus'))

        # Pillar 3: Late Momentum (t20-t30) & Tactical Dynamics
        f['net_hp_trend'] = float((p1_tot - p1_t20 * 6.0) - (p2_tot - p2_t20 * 6.0))
        f['spikes_diff'] = float(acts['p1_spikes'] - acts['p2_spikes'])
        f['switches_diff'] = float(acts['p1_sw'] - acts['p2_sw'])
        f['heals_diff'] = float(acts['p1_heals'] - acts['p2_heals'])
        f['setups_diff'] = float(acts['p1_setups'] - acts['p2_setups'])
        f['snorlax_diff'] = float('snorlax' in p1_names and p1_hp.get('snorlax', 1.0) > 0) - float('snorlax' in p2_hp and p2_hp.get('snorlax', 1.0) > 0)

        rows.append(f)
    return pd.DataFrame(rows)

print("Extracting essential features from training data...")
train_df = create_simple_features(train_data)
print(f"> Features extracted: {train_df.shape[0]} rows, {train_df.shape[1]} columns.")
train_df.head()

Extracting essential features from training data...


Extracting essential features:   0%|          | 0/10000 [00:00<?, ?it/s]

> Features extracted: 10000 rows, 31 columns.


,battle_id,player_won,p1_mean_hp,p1_std_hp,p1_mean_atk,p1_std_atk,p1_mean_def,p1_std_def,p1_mean_spa,p1_std_spa,...,act_hp_p1,act_hp_p2,act_type_adv,act_status_diff,net_hp_trend,spikes_diff,switches_diff,heals_diff,setups_diff,snorlax_diff
0,0,0,83.000000,14.719601,88.000000,27.682726,91.333333,28.235124,94.833333,28.927592,...,1.00000,0.850000,3.75,-1.0,5.010000,-1.0,0.0,1.0,-1.0,-1.0
1,1,0,91.666667,33.499585,83.333333,15.184056,99.166667,33.839901,75.833333,24.051449,...,0.00000,0.740000,-0.25,-1.0,0.240000,1.0,-2.0,0.0,0.0,0.0
2,2,0,95.000000,33.040379,113.166667,18.215531,97.500000,39.449335,77.500000,24.109127,...,0.18799,0.937337,0.00,1.0,-3.427012,2.0,1.0,0.0,4.0,-1.0
3,3,0,119.166667,69.846061,75.000000,32.145503,98.333333,54.211520,75.000000,26.299556,...,0.54000,0.950000,0.00,0.0,0.770000,0.0,0.0,7.0,1.0,0.0
4,4,0,105.166667,30.140873,89.500000,13.400871,95.333333,47.737709,94.166667,27.751376,...,0.00000,0.000000,-0.50,0.0,0.540000,0.0,-1.0,-1.0,-3.0,0.0


## 3. Model Architecture & Assembly (Multi-Model Ensemble)

To avoid the capacity constraints of a single linear model or single decision tree, we employ an ensemble of the **3 leading GBDT algorithms**:

1. **HistGradientBoostingClassifier** (Scikit-Learn): Discretizes continuous values into 256 integer bins; exceptionally fast and robust against extreme values.
2. **LGBMClassifier** (LightGBM): Asymmetric *leaf-wise* tree growth with feature and row subsampling; captures complex nonlinear interactions.
3. **XGBClassifier** (XGBoost): Balanced *depth-wise* tree growth with exact second-order Taylor (Hessian) gradient updates.

### Validation & Blending Strategy
* **Stratified 5-Fold Cross-Validation**: Splits data into 5 balanced folds and tracks **Out-Of-Fold (OOF)** predictions to completely eliminate data leakage.
* **Optimal Blending Weights**: Uses SciPy `minimize` to find the optimal convex combination $w_1 P_{	ext{Hist}} + w_2 P_{	ext{LGBM}} + w_3 P_{	ext{XGB}}$.
* **Decision Threshold Tuning**: Calibrates the optimal threshold $t^*$ on OOF probabilities to directly maximize classification Accuracy.
* **Final Model Training**: Refits all 3 models on 100% of training data (10,000 battles) for maximum test generalization.

In [4]:
%pip install xgboost
from sklearn.ensemble import HistGradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from scipy.optimize import minimize

# Separate features (X) and target variable (y)
features = [col for col in train_df.columns if col not in ['battle_id', 'player_won']]
X = train_df[features].astype(np.float32).values
y = train_df['player_won'].values

print(f"Training dataset ready: {X.shape[0]} samples with {X.shape[1]} essential features.")

# Initialize the 3 diverse GBDT models
m1 = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.04, max_leaf_nodes=35,
    min_samples_leaf=20, l2_regularization=2.0, random_state=42
)
m2 = LGBMClassifier(
    n_estimators=300, learning_rate=0.04, num_leaves=31,
    min_child_samples=25, reg_alpha=1.0, reg_lambda=2.0,
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1
)
m3 = XGBClassifier(
    n_estimators=300, learning_rate=0.04, max_depth=5,
    min_child_weight=20, reg_alpha=1.0, reg_lambda=2.0,
    subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='logloss'
)

models = {'HistGB': m1, 'LightGBM': m2, 'XGBoost': m3}

# Stratified 5-Fold Cross-Validation with Out-Of-Fold (OOF) tracking
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = {name: np.zeros(len(y), dtype=np.float32) for name in models}

print("\nStarting 5-Fold Cross-Validation for all 3 models...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X[train_idx], y[train_idx]
    X_va, y_va = X[val_idx], y[val_idx]
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        oof_preds[name][val_idx] = model.predict_proba(X_va)[:, 1]
    print(f"  > Fold {fold+1}/5 evaluated.")

print("\n--- Individual Model OOF Accuracy (default threshold 0.50) ---")
for name in models:
    acc = accuracy_score(y, oof_preds[name] > 0.5)
    print(f"  {name}: {acc*100:.2f}%")

# Assembly: Optimize ensemble blend weights on OOF predictions
def blend_loss(w):
    w_norm = np.array(w) / np.sum(w)
    prob = w_norm[0]*oof_preds['HistGB'] + w_norm[1]*oof_preds['LightGBM'] + w_norm[2]*oof_preds['XGBoost']
    return -accuracy_score(y, prob > 0.5)

res = minimize(blend_loss, [0.33, 0.33, 0.34], bounds=[(0, 1), (0, 1), (0, 1)])
opt_weights = res.x / np.sum(res.x)

print("\n--- Optimal Assembly Blend Weights ---")
print(f"  HistGB:   {opt_weights[0]:.3f}")
print(f"  LightGBM: {opt_weights[1]:.3f}")
print(f"  XGBoost:  {opt_weights[2]:.3f}")

oof_blend = (opt_weights[0] * oof_preds['HistGB'] +
             opt_weights[1] * oof_preds['LightGBM'] +
             opt_weights[2] * oof_preds['XGBoost'])

# Threshold Optimization
best_threshold = 0.5
best_acc = accuracy_score(y, oof_blend > 0.5)
for t in np.linspace(0.46, 0.54, 81):
    acc = accuracy_score(y, oof_blend > t)
    if acc > best_acc:
        best_acc = acc
        best_threshold = t

print(f"\nEnsemble Blend OOF Accuracy (standard 0.50 threshold):    {accuracy_score(y, oof_blend > 0.5)*100:.2f}%")
print(f"Ensemble Blend OOF Accuracy (calibrated {best_threshold:.4f} threshold): {best_acc*100:.2f}%")

blend_acc = accuracy_score(y, oof_blend > 0.5)
print(f"Combined ensemble accuracy: {blend_acc*100:.2f}%")

# Final Model Training on 100% of Training Data (10,000 battles)
print("\nTraining final models on 100% of training data...")
for name, model in models.items():
    model.fit(X, y)
    print(f"  > {name} ready for inference.")
print("All 3 models trained on full dataset!")



Note: you may need to restart the kernel to use updated packages.
Training dataset ready: 10000 samples with 29 essential features.

Starting 5-Fold Cross-Validation for all 3 models...
  > Fold 1/5 evaluated.
  > Fold 2/5 evaluated.
  > Fold 3/5 evaluated.
  > Fold 4/5 evaluated.
  > Fold 5/5 evaluated.

--- Individual Model OOF Accuracy (default threshold 0.50) ---
  HistGB: 66.61%
  LightGBM: 66.11%
  XGBoost: 66.48%

--- Optimal Assembly Blend Weights ---
  HistGB:   0.330
  LightGBM: 0.330
  XGBoost:  0.340

Ensemble Blend OOF Accuracy (standard 0.50 threshold):    66.70%
Ensemble Blend OOF Accuracy (calibrated 0.5270 threshold): 66.94%
Combined ensemble accuracy: 66.70%

Training final models on 100% of training data...
  > HistGB ready for inference.
  > LightGBM ready for inference.
  > XGBoost ready for inference.
All 3 models trained on full dataset!


## 4. Creating and Validating the Submission File (`submission.csv`)

In this final step:
1. Extract the essential features on test data (`test_data`).
2. Generate predicted probabilities from all 3 trained models.
3. Blend the predictions using optimal Assembly weights and apply the calibrated decision threshold.
4. Align the predictions to the `sample_submission.csv` template preserving battle IDs.
5. Save and validate the `submission.csv` file.

In [5]:
# 1. Extract essential features on test set
print("Extracting essential features from test data...")
test_df = create_simple_features(test_data)
X_test = test_df[features].astype(np.float32).values

# 2. Predict probabilities with all 3 models and apply ensemble blend
print("Generating blended ensemble predictions...")
p1_prob = models['HistGB'].predict_proba(X_test)[:, 1]
p2_prob = models['LightGBM'].predict_proba(X_test)[:, 1]
p3_prob = models['XGBoost'].predict_proba(X_test)[:, 1]

test_prob_blend = opt_weights[0] * p1_prob + opt_weights[1] * p2_prob + opt_weights[2] * p3_prob

# 3. Apply calibrated decision threshold
test_predictions = (test_prob_blend > best_threshold).astype(int)

# 4. Create compliant submission DataFrame
sample_sub_path = resolve_path('sample_submission.csv')
submission_df = pd.read_csv(sample_sub_path)

# Map predictions safely by battle_id
predictions_by_id = pd.Series(test_predictions, index=test_df['battle_id'])
submission_df['player_won'] = submission_df['battle_id'].map(predictions_by_id).astype(int)

# 5. Save submission file to disk
submission_df.to_csv('submission.csv', index=False)
print("\n> 'submission.csv' file created successfully!")
print(f"Submission shape: {submission_df.shape} (exactly 5,000 rows and 2 columns)")
print("\nPredicted class distribution:")
print(submission_df['player_won'].value_counts())
print("\nFirst 10 rows of submission.csv:")
print(submission_df.head(10))

Extracting essential features from test data...


Extracting essential features:   0%|          | 0/5000 [00:00<?, ?it/s]

Generating blended ensemble predictions...

> 'submission.csv' file created successfully!
Submission shape: (5000, 2) (exactly 5,000 rows and 2 columns)

Predicted class distribution:
player_won
0    2679
1    2321
Name: count, dtype: int64

First 10 rows of submission.csv:
   battle_id  player_won
0          0           1
1          1           0
2          2           0
3          3           0
4          4           1
5          5           0
6          6           1
7          7           0
8          8           0
9          9           0


## 5. How to Submit Your Results on Kaggle

Once `submission.csv` is generated, here is how to submit it to the Leaderboard:

---

#### Method A: Direct Notebook Submission on Kaggle (Recommended)
1. **Save Version**: Click the **"Save Version"** button in the top-right corner of the Kaggle notebook editor.
2. **Execute**: Select **"Save & Run All (Commit)"** and click **"Save"**. The notebook will execute the entire pipeline in ~1 minute.
3. **Open Viewer**: When the commit run completes, open the saved version page.
4. **Submit**: In the **Output** tab, locate `submission.csv` and click **Submit to Competition**.

---

#### Method B: Manual Upload
1. Download the generated `submission.csv` file to your computer.
2. Navigate to the competition [Leaderboard Page](https://www.kaggle.com/competitions/fds-pokemon-battles-prediction-2026/leaderboard?).
3. Click on the **"Submit Predictions"** button and upload your file.

Good luck in the competition! 🚀